# RAAZICID donation photo classifier (PyTorch, MobileNetV2)

Trains the model that suggests a donation's category from its photo (Phase 5).

**Data (option C):**
1. **Training data:** public photos from **Open Images V7** (Google, CC BY 2.0), picked by matching
   Open Images labels to our 12 categories.
2. **Real-world test set:** your own phone photos in Google Drive, `MyDrive/raazicid/own_test/<category>/`,
   20–30 per category. These are never used for training; they show how the model does on real donations.

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. About 30–60 minutes.
At the end, `raazicid_classifier.pt` and `labels.json` are saved to `MyDrive/raazicid/model/`.
Copy both into `backend/models/`.

In [ ]:
# 1. Settings
PER_CLASS = 300          # photos per category from Open Images (more = better, slower)
IMG_SIZE = 224
SEED = 42
DRIVE_DIR = "/content/drive/MyDrive/raazicid"
DATA_DIR = "/content/data"            # downloaded training photos (not saved to Drive)

from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(f"{DRIVE_DIR}/model", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/own_test", exist_ok=True)
print("Put your own test photos in", f"{DRIVE_DIR}/own_test/<category>/")

## 2. Our 12 categories → Open Images labels

`id` must match the `categories` table (backend/app/seed/data.py). Each category lists Open Images
display names; names that do not exist in Open Images are reported and skipped in the next cell.
`PRIORITY` decides the category when a photo has labels from several (e.g. a *Coat* is Winter support, not Clothes).

In [ ]:
CATEGORIES = {
    # slug: (category id, Open Images display names)
    "food":             (1,  ["Food", "Dish", "Fast food", "Baked goods", "Bread", "Cuisine", "Meal", "Lunch", "Dinner"]),
    "grocery_ration":   (2,  ["Flour", "Rice", "Lentil", "Sugar", "Cooking oil", "Packaged goods", "Sack", "Grain", "Legume", "Canned food", "Pasta"]),
    "clothes":          (3,  ["Clothing", "Shirt", "T-shirt", "Jeans", "Dress", "Trousers", "Skirt", "Shorts", "Blouse", "Kurta"]),
    "blankets":         (4,  ["Blanket", "Quilt", "Duvet", "Bed sheet", "Linens", "Comforter"]),
    "winter_support":   (5,  ["Coat", "Jacket", "Sweater", "Scarf", "Glove", "Knit cap", "Hoodie", "Beanie", "Space heater", "Overcoat"]),
    "hygiene":          (6,  ["Soap", "Toothbrush", "Toothpaste", "Shampoo", "Towel", "Toilet paper", "Hand sanitizer", "Sanitary napkin", "Bath towel"]),
    "baby":             (7,  ["Diaper", "Baby bottle", "Infant bed", "Baby carriage", "Stroller", "Teddy bear", "Baby toys", "Bib", "Pacifier"]),
    "household":        (8,  ["Cookware and bakeware", "Kitchen utensil", "Plate", "Bowl", "Frying pan", "Kettle", "Bucket", "Mattress", "Pillow", "Tableware", "Pressure cooker"]),
    "school_supplies":  (9,  ["Pencil", "Pen", "Crayon", "Notebook", "Backpack", "School bag", "Ruler", "Eraser", "Pencil case", "Stationery", "Marker pen"]),
    "books":            (10, ["Book", "Textbook", "Paperback", "Hardcover", "Comic book", "Novel"]),
    "mobility_aids":    (11, ["Wheelchair", "Crutch", "Walker", "Walking stick", "Cane", "Mobility scooter"]),
    "emergency_relief": (12, ["Tent", "First aid kit", "Bottled water", "Water bottle", "Tarpaulin", "Flashlight", "Jerrycan", "Emergency blanket"]),
}
# Most specific first
PRIORITY = ["mobility_aids", "baby", "school_supplies", "books", "hygiene", "emergency_relief", "blankets",
            "winter_support", "grocery_ration", "household", "food", "clothes"]
assert set(PRIORITY) == set(CATEGORIES)

In [ ]:
# 3. Check which names exist in Open Images V7
import pandas as pd
classes = pd.read_csv("https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions.csv")
name_to_mid = {}
for mid, name in zip(classes["LabelName"], classes["DisplayName"]):
    name_to_mid.setdefault(name.lower(), mid)

MID_TO_SLUG = {}
for slug, (_, names) in CATEGORIES.items():
    found = [n for n in names if n.lower() in name_to_mid]
    missing = [n for n in names if n.lower() not in name_to_mid]
    for n in found:
        MID_TO_SLUG.setdefault(name_to_mid[n.lower()], slug)
    print(f"{slug:17s} found {len(found):2d}: {', '.join(found)}" + (f"   | not in Open Images: {', '.join(missing)}" if missing else ""))

In [ ]:
# 4. Find photos for each category in the human-verified image labels.
#    The files are big (train is several GB), so they are read in chunks and we stop once every
#    category has enough candidates. Common labels (Clothing, Food) are capped so memory stays small.
import random
from collections import defaultdict

random.seed(SEED)
WANT = PER_CLASS * 3            # collect extra candidates; some downloads fail
LABEL_FILES = [
    ("validation", "https://storage.googleapis.com/openimages/v7/oidv7-val-annotations-human-imagelabels.csv"),
    ("test",       "https://storage.googleapis.com/openimages/v7/oidv7-test-annotations-human-imagelabels.csv"),
    ("train",      "https://storage.googleapis.com/openimages/v7/oidv7-train-annotations-human-imagelabels.csv"),
]
by_slug = defaultdict(list)     # category -> [(split, image_id)]
seen = set()

def full():
    return all(len(by_slug[s]) >= WANT for s in CATEGORIES)

for split, url in LABEL_FILES:
    print("Reading", split, "labels…")
    for chunk in pd.read_csv(url, usecols=["ImageID", "LabelName", "Confidence"], chunksize=2_000_000):
        hit = chunk[(chunk["Confidence"] == 1) & chunk["LabelName"].isin(MID_TO_SLUG.keys())]
        # all our labels of each photo -> ONE category (most specific first)
        for img, mids in hit.groupby("ImageID")["LabelName"]:
            if (split, img) in seen:
                continue
            seen.add((split, img))
            slug = min((MID_TO_SLUG[m] for m in mids), key=PRIORITY.index)
            if len(by_slug[slug]) < WANT:
                by_slug[slug].append((split, img))
        if full():
            break
    print({s: len(by_slug[s]) for s in PRIORITY})
    if full():
        break

for s in PRIORITY:
    random.shuffle(by_slug[s])
    print(f"{s:17s} {len(by_slug[s])} candidate photos")

In [ ]:
# 5. Download and shrink the photos (public Open Images bucket)
import io, requests
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

def fetch(args):
    slug, (split, img_id) = args
    out = f"{DATA_DIR}/{slug}/{img_id}.jpg"
    if os.path.exists(out):
        return True
    try:
        r = requests.get(f"https://open-images-dataset.s3.amazonaws.com/{split}/{img_id}.jpg", timeout=20)
        r.raise_for_status()
        im = Image.open(io.BytesIO(r.content)).convert("RGB")
        im.thumbnail((320, 320))
        im.save(out, quality=88)
        return True
    except Exception:
        return False

for slug in PRIORITY:
    os.makedirs(f"{DATA_DIR}/{slug}", exist_ok=True)
    have = len(os.listdir(f"{DATA_DIR}/{slug}"))
    queue = by_slug[slug]
    i = 0
    while have < PER_CLASS and i < len(queue):
        batch = queue[i:i + (PER_CLASS - have) + 20]
        i += len(batch)
        with ThreadPoolExecutor(16) as pool:
            list(pool.map(fetch, [(slug, k) for k in batch]))
        have = len(os.listdir(f"{DATA_DIR}/{slug}"))
    print(f"{slug:17s} {have} photos" + ("   <-- few photos: add more own photos for this one" if have < 100 else ""))

## 6. Train / validation / test split

Open Images photos: 70% train, 15% validation (to pick the best epoch), 15% test (final score).
Your own photos in Drive are a **separate** real-world test set.

In [ ]:
import glob, numpy as np, torch
from sklearn.model_selection import train_test_split

SLUGS = sorted(CATEGORIES, key=lambda s: CATEGORIES[s][0])     # index order = category id order
files, labels = [], []
for idx, slug in enumerate(SLUGS):
    for f in glob.glob(f"{DATA_DIR}/{slug}/*.jpg"):
        files.append(f); labels.append(idx)
files, labels = np.array(files), np.array(labels)
f_train, f_tmp, y_train, y_tmp = train_test_split(files, labels, test_size=0.30, stratify=labels, random_state=SEED)
f_val, f_test, y_val, y_test = train_test_split(f_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print(len(f_train), "train /", len(f_val), "validation /", len(f_test), "test")

own_files, own_labels = [], []
for idx, slug in enumerate(SLUGS):
    for f in glob.glob(f"{DRIVE_DIR}/own_test/{slug}/*"):
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            own_files.append(f); own_labels.append(idx)
print(len(own_files), "own test photos")

In [ ]:
# 7. Model: MobileNetV2 pretrained on ImageNet, new last layer for our 12 categories
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05), transforms.RandomRotation(10),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMG_SIZE), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class Photos(Dataset):
    def __init__(self, files, labels, tf): self.files, self.labels, self.tf = files, labels, tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        return self.tf(Image.open(self.files[i]).convert("RGB")), int(self.labels[i])

dl = lambda f, y, tf, shuffle: DataLoader(Photos(f, y, tf), batch_size=64, shuffle=shuffle, num_workers=2)
train_dl, val_dl, test_dl = dl(f_train, y_train, train_tf, True), dl(f_val, y_val, eval_tf, False), dl(f_test, y_test, eval_tf, False)

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2)
model.classifier[1] = nn.Linear(model.last_channel, len(SLUGS))
model = model.to(device)

# Some categories have fewer photos: weight the loss so they still count
counts = np.bincount(y_train, minlength=len(SLUGS))
weights = torch.tensor(counts.sum() / (len(SLUGS) * np.maximum(counts, 1)), dtype=torch.float32, device=device)
loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

In [ ]:
# 8. Train: first only the new layer, then the whole network with a small learning rate
import copy, time

def run_epoch(loader, train, opt=None):
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item() * len(y); total += len(y); correct += (out.argmax(1) == y).sum().item()
    return loss_sum / total, correct / total

history, best_acc, best_state = [], 0.0, None
for stage, (epochs, lr, freeze) in enumerate([(3, 1e-3, True), (10, 1e-4, False)], 1):
    for p in model.features.parameters():
        p.requires_grad = not freeze
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(epochs):
        t = time.time()
        tr_loss, tr_acc = run_epoch(train_dl, True, opt)
        va_loss, va_acc = run_epoch(val_dl, False)
        sched.step()
        history.append({"stage": stage, "train_acc": tr_acc, "val_acc": va_acc, "train_loss": tr_loss, "val_loss": va_loss})
        if va_acc > best_acc:
            best_acc, best_state = va_acc, copy.deepcopy(model.state_dict())
        print(f"stage {stage} epoch {ep + 1}: train acc {tr_acc:.3f}  val acc {va_acc:.3f}  ({time.time() - t:.0f}s)")
model.load_state_dict(best_state)
print("Best validation accuracy:", round(best_acc, 3))

In [ ]:
# 9. Evaluate: accuracy, top-3 accuracy, per-category report and confusion matrix
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

def predict(files, tf=eval_tf):
    model.eval(); probs = []
    with torch.no_grad():
        for i in range(0, len(files), 64):
            x = torch.stack([tf(Image.open(f).convert("RGB")) for f in files[i:i + 64]]).to(device)
            probs.append(torch.softmax(model(x), 1).cpu())
    return torch.cat(probs).numpy()

def evaluate(name, files, labels):
    if len(files) == 0:
        print(f"{name}: no photos"); return None
    p = predict(list(files)); y = np.array(labels); pred = p.argmax(1)
    top3 = np.mean([y[i] in np.argsort(-p[i])[:3] for i in range(len(y))])
    acc = float((pred == y).mean())
    print(f"\n== {name}: accuracy {acc:.3f}, top-3 accuracy {top3:.3f}, {len(y)} photos ==")
    present = sorted(set(y) | set(pred))
    print(classification_report(y, pred, labels=present, target_names=[SLUGS[i] for i in present], zero_division=0))
    cm = confusion_matrix(y, pred, labels=range(len(SLUGS)))
    fig, ax = plt.subplots(figsize=(10, 9))
    ConfusionMatrixDisplay(cm, display_labels=SLUGS).plot(ax=ax, xticks_rotation=60, colorbar=False)
    ax.set_title(f"{name} (accuracy {acc:.1%})"); plt.tight_layout()
    path = f"{DRIVE_DIR}/model/confusion_{name.replace(' ', '_').lower()}.png"
    plt.savefig(path, dpi=150); plt.show()
    report = classification_report(y, pred, labels=present, target_names=[SLUGS[i] for i in present], zero_division=0, output_dict=True)
    return {"photos": int(len(y)), "accuracy": round(acc, 4), "top3_accuracy": round(float(top3), 4), "per_category": report}

metrics = {"open_images_test": evaluate("Open Images test", f_test, y_test),
           "own_photos": evaluate("Own photos", own_files, own_labels)}

h = pd.DataFrame(history)
h[["train_acc", "val_acc"]].plot(title="Accuracy per epoch"); plt.xlabel("epoch")
plt.savefig(f"{DRIVE_DIR}/model/training_curve.png", dpi=150); plt.show()

In [ ]:
# 10. Export for the backend: TorchScript model + labels.json
import json, datetime
model.eval().cpu()
scripted = torch.jit.trace(model, torch.randn(1, 3, IMG_SIZE, IMG_SIZE))
scripted.save(f"{DRIVE_DIR}/model/raazicid_classifier.pt")
info = {
    "version": datetime.datetime.utcnow().strftime("mobilenetv2-%Y%m%d-%H%M"),
    "architecture": "MobileNetV2 (ImageNet weights, fine-tuned)",
    "input_size": IMG_SIZE, "resize": 256, "mean": MEAN, "std": STD,
    "categories": [{"index": i, "slug": s, "category_id": CATEGORIES[s][0]} for i, s in enumerate(SLUGS)],
    "train_photos": int(len(f_train)), "best_val_accuracy": round(best_acc, 4),
    "metrics": metrics, "history": history,
    "data": "Open Images V7 (CC BY 2.0) for training; own photos only for testing",
}
json.dump(info, open(f"{DRIVE_DIR}/model/labels.json", "w"), indent=2, default=float)
print("Saved to", f"{DRIVE_DIR}/model/:", os.listdir(f"{DRIVE_DIR}/model"))

In [ ]:
# 11. Try it on any photo (upload one)
from google.colab import files as colab_files
up = colab_files.upload()
for name in up:
    p = predict([name])[0]
    for i in np.argsort(-p)[:3]:
        print(f"{SLUGS[i]:17s} {p[i]:.1%}")